# Random Forest from Scratch

**Interview-focused reference notebook** -- Bagging, random feature subsets, OOB error, feature importance.

## Core Intuition

Random Forest = ensemble of **decorrelated** decision trees.

Two sources of randomness:
1. **Bagging** (Bootstrap Aggregating): each tree trains on a bootstrap sample (sample with replacement)
2. **Random feature subsets**: at each split, only consider $m \approx \sqrt{d}$ random features

**Why it works:** Averaging $B$ trees reduces variance by roughly $1/B$ (if they're independent). Bagging + feature randomness makes the trees less correlated, so the variance reduction is more effective.

**Bias-variance:** Individual trees have low bias but high variance. The ensemble maintains the low bias while drastically reducing variance.

$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$$

where $\rho$ is the average pairwise correlation between trees. Feature randomness reduces $\rho$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Decision Tree Base Learner

We reuse a simplified decision tree (same CART logic from notebook 06) with an added parameter for random feature selection at each split.

In [ ]:
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 value=None, n_samples=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.n_samples = n_samples
    
    def is_leaf(self):
        return self.value is not None


class DecisionTree:
    """Decision tree with optional random feature selection (for RF)."""
    
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 max_features=None, task='classification'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features  # number of features to consider at each split
        self.task = task
    
    def _impurity(self, y):
        if self.task == 'classification':
            classes, counts = np.unique(y, return_counts=True)
            probs = counts / len(y)
            return 1 - np.sum(probs ** 2)  # Gini
        else:
            return np.var(y)  # Variance for regression
    
    def _leaf_value(self, y):
        if self.task == 'classification':
            classes, counts = np.unique(y, return_counts=True)
            return classes[np.argmax(counts)]
        else:
            return np.mean(y)
    
    def _best_split(self, X, y):
        best_gain = -1
        best_feature = None
        best_threshold = None
        n, d = X.shape
        parent_impurity = self._impurity(y)
        
        # Random feature selection (key RF ingredient)
        if self.max_features is not None:
            features = np.random.choice(d, min(self.max_features, d), replace=False)
        else:
            features = np.arange(d)
        
        for feature in features:
            thresholds = np.unique(X[:, feature])
            if len(thresholds) > 1:
                thresholds = (thresholds[:-1] + thresholds[1:]) / 2
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                n_left = left_mask.sum()
                n_right = right_mask.sum()
                
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue
                
                child_impurity = (n_left/n * self._impurity(y[left_mask]) + 
                                  n_right/n * self._impurity(y[right_mask]))
                gain = parent_impurity - child_impurity
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        n_samples = len(y)
        
        if (n_samples < self.min_samples_split or
            (self.max_depth is not None and depth >= self.max_depth) or
            (self.task == 'classification' and len(np.unique(y)) == 1) or
            (self.task == 'regression' and np.var(y) < 1e-10)):
            return TreeNode(value=self._leaf_value(y), n_samples=n_samples)
        
        best_feature, best_threshold, best_gain = self._best_split(X, y)
        
        if best_feature is None or best_gain <= 0:
            return TreeNode(value=self._leaf_value(y), n_samples=n_samples)
        
        left_mask = X[:, best_feature] <= best_threshold
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        
        return TreeNode(feature=best_feature, threshold=best_threshold,
                        left=left_child, right=right_child, n_samples=n_samples)
    
    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self
    
    def _predict_single(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)
    
    def predict(self, X):
        return np.array([self._predict_single(x, self.root) for x in X])

print("Decision tree base learner ready.")

## 2. Synthetic Data

In [ ]:
from sklearn.datasets import make_moons, make_classification, make_blobs

# Classification datasets
X_moons, y_moons = make_moons(n_samples=400, noise=0.25, random_state=42)
X_class, y_class = make_classification(n_samples=500, n_features=10, n_informative=5,
                                        n_redundant=2, random_state=42)

# Regression dataset
X_reg = np.sort(np.random.uniform(0, 10, 300)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + 0.3 * np.random.randn(300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdBu', s=20, alpha=0.7, edgecolors='k', linewidths=0.5)
axes[0].set_title('Moons Dataset (Classification)')
axes[1].scatter(X_reg, y_reg, s=15, alpha=0.6)
axes[1].set_title('Regression Dataset')
plt.tight_layout()
plt.show()

## 3. Bagging (Bootstrap Aggregating)

Bagging trains each model on a bootstrap sample (sample with replacement, same size as original). The key insight is that each bootstrap sample contains about 63.2% of the unique training samples.

In [ ]:
class BaggingClassifier:
    """Bagging ensemble -- bootstrap aggregating."""
    
    def __init__(self, n_estimators=10, max_depth=None, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
        self.oob_indices = []  # for OOB error
    
    def fit(self, X, y):
        n = X.shape[0]
        self.trees = []
        self.oob_indices = []
        
        for _ in range(self.n_estimators):
            # Bootstrap sample
            boot_indices = np.random.choice(n, size=n, replace=True)
            oob_mask = np.ones(n, dtype=bool)
            oob_mask[boot_indices] = False
            self.oob_indices.append(np.where(oob_mask)[0])
            
            X_boot = X[boot_indices]
            y_boot = y[boot_indices]
            
            tree = DecisionTree(max_depth=self.max_depth, max_features=self.max_features,
                                task='classification')
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
        
        self.X_train_ = X
        self.y_train_ = y
        return self
    
    def predict(self, X):
        """Majority vote across all trees."""
        all_preds = np.array([tree.predict(X) for tree in self.trees])  # (n_trees, n_samples)
        # Majority vote
        predictions = []
        for i in range(X.shape[0]):
            votes = all_preds[:, i]
            predictions.append(Counter(votes).most_common(1)[0][0])
        return np.array(predictions)

# Verify bootstrap sample coverage
n = 1000
boot = np.random.choice(n, size=n, replace=True)
unique_fraction = len(np.unique(boot)) / n
print(f"Bootstrap sample: {unique_fraction:.3f} unique samples (theoretical: 1-1/e = {1-1/np.e:.3f})")

## 4. Random Forest Implementation

In [ ]:
class RandomForestClassifier:
    """Random Forest = Bagging + Random Feature Subsets."""
    
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt',
                 min_samples_split=2, min_samples_leaf=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.trees = []
        self.oob_indices = []
    
    def _get_max_features(self, d):
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(d)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(d)))
        elif isinstance(self.max_features, int):
            return self.max_features
        elif isinstance(self.max_features, float):
            return max(1, int(self.max_features * d))
        else:
            return d  # use all features
    
    def fit(self, X, y):
        n, d = X.shape
        max_feat = self._get_max_features(d)
        self.classes_ = np.unique(y)
        self.trees = []
        self.oob_indices = []
        
        for i in range(self.n_estimators):
            # Bootstrap sample
            boot_indices = np.random.choice(n, size=n, replace=True)
            oob_mask = np.ones(n, dtype=bool)
            oob_mask[np.unique(boot_indices)] = False
            self.oob_indices.append(np.where(oob_mask)[0])
            
            X_boot = X[boot_indices]
            y_boot = y[boot_indices]
            
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_feat,
                task='classification'
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
        
        self.X_train_ = X
        self.y_train_ = y
        self.n_features_ = d
        return self
    
    def predict(self, X):
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        predictions = []
        for i in range(X.shape[0]):
            votes = all_preds[:, i]
            predictions.append(Counter(votes).most_common(1)[0][0])
        return np.array(predictions)
    
    def oob_score(self):
        """Compute out-of-bag error estimate."""
        n = self.X_train_.shape[0]
        oob_predictions = {}
        
        for tree_idx, (tree, oob_idx) in enumerate(zip(self.trees, self.oob_indices)):
            if len(oob_idx) == 0:
                continue
            preds = tree.predict(self.X_train_[oob_idx])
            for sample_idx, pred in zip(oob_idx, preds):
                if sample_idx not in oob_predictions:
                    oob_predictions[sample_idx] = []
                oob_predictions[sample_idx].append(pred)
        
        # Majority vote for each OOB sample
        correct = 0
        total = 0
        for idx, preds in oob_predictions.items():
            majority = Counter(preds).most_common(1)[0][0]
            if majority == self.y_train_[idx]:
                correct += 1
            total += 1
        
        return correct / total if total > 0 else 0

# Train Random Forest
rf = RandomForestClassifier(n_estimators=50, max_depth=10, max_features='sqrt').fit(X_moons, y_moons)
acc = np.mean(rf.predict(X_moons) == y_moons)
oob = rf.oob_score()
print(f"Random Forest accuracy: {acc:.4f}")
print(f"OOB score: {oob:.4f}")

## 5. Random Forest for Regression

In [ ]:
class RandomForestRegressor:
    """Random Forest for regression -- average predictions instead of vote."""
    
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt'):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
    
    def _get_max_features(self, d):
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(d)))
        elif isinstance(self.max_features, int):
            return self.max_features
        return d
    
    def fit(self, X, y):
        n, d = X.shape
        max_feat = self._get_max_features(d)
        self.trees = []
        
        for _ in range(self.n_estimators):
            boot_indices = np.random.choice(n, size=n, replace=True)
            tree = DecisionTree(max_depth=self.max_depth, max_features=max_feat, task='regression')
            tree.fit(X[boot_indices], y[boot_indices])
            self.trees.append(tree)
        return self
    
    def predict(self, X):
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(all_preds, axis=0)  # average for regression

rf_reg = RandomForestRegressor(n_estimators=50, max_depth=8).fit(X_reg, y_reg)
y_pred = rf_reg.predict(X_reg)
mse = np.mean((y_pred - y_reg) ** 2)
print(f"RF Regressor MSE: {mse:.4f}")

## 6. Single Tree vs Random Forest: Decision Boundaries

In [ ]:
def plot_boundary(model, X, y, ax, title=''):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=20, alpha=0.7, edgecolors='k', linewidths=0.5)
    ax.set_title(title)

# Train different models
single_tree = DecisionTree(max_depth=15, task='classification').fit(X_moons, y_moons)
bagging_model = BaggingClassifier(n_estimators=50, max_depth=15).fit(X_moons, y_moons)
rf_model = RandomForestClassifier(n_estimators=50, max_depth=15, max_features='sqrt').fit(X_moons, y_moons)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_boundary(single_tree, X_moons, y_moons, axes[0], 
              f'Single Tree (acc={np.mean(single_tree.predict(X_moons)==y_moons):.3f})')
plot_boundary(bagging_model, X_moons, y_moons, axes[1],
              f'Bagging (acc={np.mean(bagging_model.predict(X_moons)==y_moons):.3f})')
plot_boundary(rf_model, X_moons, y_moons, axes[2],
              f'Random Forest (acc={np.mean(rf_model.predict(X_moons)==y_moons):.3f})')

plt.suptitle('Single Tree vs Bagging vs Random Forest', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("Single tree: jagged, overfit boundaries.")
print("Bagging: smoother due to averaging, but trees are correlated.")
print("Random Forest: smoothest -- feature randomness decorrelates the trees.")

## 7. Variance Reduction with Ensemble Size

In [ ]:
# Show how accuracy improves and variance decreases with more trees
n_estimator_range = [1, 2, 5, 10, 20, 30, 50, 75, 100]
n_trials = 10

mean_accs = []
std_accs = []
oob_scores = []

for n_est in n_estimator_range:
    trial_accs = []
    trial_oob = []
    for trial in range(n_trials):
        rf_trial = RandomForestClassifier(n_estimators=n_est, max_depth=10, max_features='sqrt')
        rf_trial.fit(X_moons, y_moons)
        trial_accs.append(np.mean(rf_trial.predict(X_moons) == y_moons))
        trial_oob.append(rf_trial.oob_score())
    
    mean_accs.append(np.mean(trial_accs))
    std_accs.append(np.std(trial_accs))
    oob_scores.append(np.mean(trial_oob))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].errorbar(n_estimator_range, mean_accs, yerr=std_accs, 
                 fmt='bo-', linewidth=2, markersize=6, capsize=4)
axes[0].set_xlabel('Number of Trees')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Ensemble Size (mean +/- std over 10 trials)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_estimator_range, oob_scores, 'ro-', linewidth=2, markersize=6)
axes[1].set_xlabel('Number of Trees')
axes[1].set_ylabel('OOB Score')
axes[1].set_title('Out-of-Bag Score Convergence')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Key observations:")
print("  - Variance (error bars) decreases as n_trees increases")
print("  - Diminishing returns beyond ~50 trees for this dataset")
print("  - OOB score converges and provides a free validation estimate")

## 8. Regression: Single Tree vs Random Forest

In [ ]:
X_plot = np.linspace(0, 10, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Single tree
single_reg = DecisionTree(max_depth=8, task='regression').fit(X_reg, y_reg)
axes[0].scatter(X_reg, y_reg, s=10, alpha=0.4, c='blue')
axes[0].plot(X_plot, single_reg.predict(X_plot), 'r-', linewidth=2)
axes[0].plot(X_plot, np.sin(X_plot), 'g--', alpha=0.5, label='True')
axes[0].set_title(f'Single Tree (MSE={np.mean((single_reg.predict(X_reg)-y_reg)**2):.3f})')
axes[0].legend()

# Show individual tree predictions
rf_demo = RandomForestRegressor(n_estimators=20, max_depth=8).fit(X_reg, y_reg)
axes[1].scatter(X_reg, y_reg, s=10, alpha=0.4, c='blue')
for tree in rf_demo.trees[:10]:
    axes[1].plot(X_plot, tree.predict(X_plot), 'gray', alpha=0.2, linewidth=1)
axes[1].plot(X_plot, rf_demo.predict(X_plot), 'r-', linewidth=2, label='RF Mean')
axes[1].plot(X_plot, np.sin(X_plot), 'g--', alpha=0.5, label='True')
axes[1].set_title('Individual Trees (gray) + RF Mean (red)')
axes[1].legend(fontsize=9)

# RF vs single tree
rf_full = RandomForestRegressor(n_estimators=50, max_depth=8).fit(X_reg, y_reg)
axes[2].scatter(X_reg, y_reg, s=10, alpha=0.4, c='blue')
axes[2].plot(X_plot, single_reg.predict(X_plot), 'orange', linewidth=1.5, alpha=0.7, label='Single Tree')
axes[2].plot(X_plot, rf_full.predict(X_plot), 'r-', linewidth=2, label='Random Forest')
axes[2].plot(X_plot, np.sin(X_plot), 'g--', alpha=0.5, label='True')
axes[2].set_title(f'RF (MSE={np.mean((rf_full.predict(X_reg)-y_reg)**2):.3f})')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 9. Feature Importance

Two common methods:
1. **Impurity-based**: sum of (weighted) impurity decreases at each split for each feature, averaged across all trees
2. **Permutation importance**: measure accuracy drop when a feature is randomly shuffled

In [ ]:
def impurity_importance(tree_node, n_features, total_samples):
    """Compute impurity-based feature importance for a single tree."""
    importances = np.zeros(n_features)
    
    def _recurse(node):
        if node.is_leaf():
            return
        # This is simplified -- uses sample counts as proxy
        n = node.n_samples
        n_left = node.left.n_samples if node.left else 0
        n_right = node.right.n_samples if node.right else 0
        importances[node.feature] += n / total_samples
        _recurse(node.left)
        _recurse(node.right)
    
    _recurse(tree_node)
    return importances


def permutation_importance(model, X, y, n_repeats=5):
    """Compute permutation importance."""
    baseline_acc = np.mean(model.predict(X) == y)
    n_features = X.shape[1]
    importances = np.zeros(n_features)
    
    for j in range(n_features):
        acc_drops = []
        for _ in range(n_repeats):
            X_permuted = X.copy()
            X_permuted[:, j] = np.random.permutation(X_permuted[:, j])
            permuted_acc = np.mean(model.predict(X_permuted) == y)
            acc_drops.append(baseline_acc - permuted_acc)
        importances[j] = np.mean(acc_drops)
    
    return importances

# Train on multi-feature dataset
rf_fi = RandomForestClassifier(n_estimators=50, max_depth=10, max_features='sqrt').fit(X_class, y_class)

# Impurity-based importance (aggregated across trees)
imp_importance = np.zeros(X_class.shape[1])
for tree in rf_fi.trees:
    imp_importance += impurity_importance(tree.root, X_class.shape[1], X_class.shape[0])
imp_importance /= len(rf_fi.trees)
if imp_importance.sum() > 0:
    imp_importance /= imp_importance.sum()

# Permutation importance
perm_importance = permutation_importance(rf_fi, X_class, y_class, n_repeats=5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

idx = np.argsort(imp_importance)[::-1]
axes[0].bar(range(len(imp_importance)), imp_importance[idx], color='steelblue')
axes[0].set_xticks(range(len(imp_importance)))
axes[0].set_xticklabels([f'F{i}' for i in idx])
axes[0].set_ylabel('Importance')
axes[0].set_title('Impurity-Based Feature Importance')
axes[0].grid(True, alpha=0.3, axis='y')

idx2 = np.argsort(perm_importance)[::-1]
axes[1].bar(range(len(perm_importance)), perm_importance[idx2], color='coral')
axes[1].set_xticks(range(len(perm_importance)))
axes[1].set_xticklabels([f'F{i}' for i in idx2])
axes[1].set_ylabel('Mean Accuracy Drop')
axes[1].set_title('Permutation Feature Importance')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print("Impurity-based importance is biased toward high-cardinality features.")
print("Permutation importance is model-agnostic and more reliable.")

## 10. Bias-Variance Decomposition Demonstration

In [ ]:
# Show that RF reduces variance compared to single tree
# Train multiple models on different bootstrap samples of the same data
n_experiments = 20
X_test_bv = np.linspace(0, 10, 100).reshape(-1, 1)

single_tree_preds = []
rf_preds = []

for exp in range(n_experiments):
    # Generate a fresh dataset (to simulate different draws from the same distribution)
    rng = np.random.RandomState(exp)
    X_train = np.sort(rng.uniform(0, 10, 200)).reshape(-1, 1)
    y_train = np.sin(X_train.ravel()) + 0.3 * rng.randn(200)
    
    # Single tree
    st = DecisionTree(max_depth=8, task='regression')
    st.fit(X_train, y_train)
    single_tree_preds.append(st.predict(X_test_bv))
    
    # Random forest
    rf_bv = RandomForestRegressor(n_estimators=30, max_depth=8)
    rf_bv.fit(X_train, y_train)
    rf_preds.append(rf_bv.predict(X_test_bv))

single_tree_preds = np.array(single_tree_preds)
rf_preds = np.array(rf_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Single tree
for pred in single_tree_preds:
    axes[0].plot(X_test_bv, pred, 'r-', alpha=0.15, linewidth=1)
axes[0].plot(X_test_bv, single_tree_preds.mean(axis=0), 'b-', linewidth=2, label='Mean prediction')
axes[0].plot(X_test_bv, np.sin(X_test_bv), 'g--', linewidth=2, label='True function')
axes[0].set_title(f'Single Tree: Variance = {single_tree_preds.var(axis=0).mean():.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Random Forest
for pred in rf_preds:
    axes[1].plot(X_test_bv, pred, 'r-', alpha=0.15, linewidth=1)
axes[1].plot(X_test_bv, rf_preds.mean(axis=0), 'b-', linewidth=2, label='Mean prediction')
axes[1].plot(X_test_bv, np.sin(X_test_bv), 'g--', linewidth=2, label='True function')
axes[1].set_title(f'Random Forest: Variance = {rf_preds.var(axis=0).mean():.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bias-Variance: Single Tree (high variance) vs RF (low variance)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Single tree avg variance: {single_tree_preds.var(axis=0).mean():.4f}")
print(f"RF avg variance:          {rf_preds.var(axis=0).mean():.4f}")
print(f"Variance reduction ratio:  {single_tree_preds.var(axis=0).mean() / rf_preds.var(axis=0).mean():.1f}x")

## 11. Sklearn Comparison

In [ ]:
from sklearn.ensemble import RandomForestClassifier as SkRF
from sklearn.ensemble import RandomForestRegressor as SkRFReg
from sklearn.tree import DecisionTreeClassifier as SkTree
from sklearn.metrics import accuracy_score, mean_squared_error

print("Classification (Moons):")
print(f"{'Model':<25} {'Accuracy':>10}")
print("-" * 37)

# Our models
our_tree = DecisionTree(max_depth=10, task='classification').fit(X_moons, y_moons)
our_rf = RandomForestClassifier(n_estimators=50, max_depth=10).fit(X_moons, y_moons)

# sklearn models
sk_tree = SkTree(max_depth=10, random_state=42).fit(X_moons, y_moons)
sk_rf = SkRF(n_estimators=50, max_depth=10, random_state=42).fit(X_moons, y_moons)

print(f"{'Our Single Tree':<25} {accuracy_score(y_moons, our_tree.predict(X_moons)):>10.4f}")
print(f"{'sklearn Single Tree':<25} {sk_tree.score(X_moons, y_moons):>10.4f}")
print(f"{'Our Random Forest':<25} {accuracy_score(y_moons, our_rf.predict(X_moons)):>10.4f}")
print(f"{'sklearn Random Forest':<25} {sk_rf.score(X_moons, y_moons):>10.4f}")
print(f"{'Our RF OOB Score':<25} {our_rf.oob_score():>10.4f}")

# Regression comparison
print("\nRegression (Sine):")
our_rf_reg = RandomForestRegressor(n_estimators=50, max_depth=8).fit(X_reg, y_reg)
sk_rf_reg = SkRFReg(n_estimators=50, max_depth=8, random_state=42).fit(X_reg, y_reg)

our_mse = mean_squared_error(y_reg, our_rf_reg.predict(X_reg))
sk_mse = mean_squared_error(y_reg, sk_rf_reg.predict(X_reg))
print(f"  Our RF MSE:     {our_mse:.4f}")
print(f"  sklearn RF MSE: {sk_mse:.4f}")

## 12. Hyperparameter Guide

In [ ]:
print("Random Forest Hyperparameter Guide:")
print("=" * 65)
print(f"{'Parameter':<25} {'Typical Value':<20} {'Effect'}")
print("-" * 65)
params = [
    ('n_estimators', '100-500', 'More = better (diminishing returns)'),
    ('max_depth', 'None or 10-30', 'Deeper = more complex trees'),
    ('max_features', 'sqrt(d) for clf', 'Less = more decorrelation'),
    ('', 'd/3 for reg', ''),
    ('min_samples_split', '2-10', 'Higher = more regularization'),
    ('min_samples_leaf', '1-5', 'Higher = smoother predictions'),
    ('bootstrap', 'True', 'False = pasting (no replacement)'),
]
for name, val, effect in params:
    print(f"{name:<25} {val:<20} {effect}")

print("\nTuning priority: n_estimators > max_features > max_depth > min_samples_leaf")

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| Why random feature subsets? | Reduces correlation between trees. Without it, all trees would split on the same dominant features, limiting variance reduction from averaging. |
| Bagging vs boosting? | Bagging: parallel, reduces variance, trees are independent. Boosting: sequential, reduces bias, each tree corrects previous errors. RF is bagging; XGBoost/AdaBoost is boosting. |
| How to tune a random forest? | Most important: n_estimators (more is better, just costs compute). Then max_features (sqrt(d) for classification, d/3 for regression). max_depth and min_samples_leaf for regularization. |
| Bias-variance decomposition? | $\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$. First term is irreducible (tree correlation). Second term shrinks with more trees. Feature randomness reduces $\rho$. |
| What is OOB error? | Each tree doesn't see ~37% of data (out-of-bag). Use those samples for a free cross-validation estimate. No need for a separate validation set! |
| RF vs single tree? | RF: much lower variance, slightly higher bias (averaging smooths predictions), better generalization, harder to interpret. |
| Can RF overfit? | Technically yes with noisy data, but much more robust than single trees. More trees never hurts (unlike boosting). |
| RF vs gradient boosting? | RF: simpler, parallelizable, robust to hyperparameters. GB: usually higher accuracy, but sensitive to hyperparameters, sequential (slower to train). |